# Pipeline Diagnostic — Find Where the Chain Breaks

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `model1_complement.pt`
   - `model2_scorer.pt`

**What this checks:**
1. **Seed hit rate** — does BM25+dense seed retrieval find any gold passage in top-5?
2. **Graph edge coverage** — are the gold chain edges (A→B) in the graph?
3. **BFS upper-bound recall** — with perfect scoring, what is the max recall the graph allows?
4. **Complement drift** — how similar is `complement(A,B)` to `m1_emb[B]`? (if ≈1.0, complement = plain passage embedding, scoring adds no value)

**Runtime: ~5 min** (reuses cached data from eval run — no recomputation needed)

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 — change to T4 GPU")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone / update repo + install deps
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")  # picks up diagnose_pipeline.py

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown")

try:
    import faiss
    print(f"faiss: OK (GPU={faiss.get_num_gpus()})")
except ImportError:
    os.system("pip install -q faiss-cpu")
    print("faiss-cpu installed")

print("Dependencies ready")

In [ ]:
# 3. Download data + checkpoints from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID, output=download_dir,
        quiet=False, use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nFiles:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")

In [ ]:
# 4. Set up file paths
import os, shutil

drive = "/kaggle/working/drive_data"
os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

for ckpt in ["model1_complement.pt", "model2_scorer.pt"]:
    src = f"{drive}/{ckpt}"
    dst = f"{WORK_DIR}/models/{ckpt}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Check which caches exist (built by eval_kaggle.ipynb)
import os

cache_dir = f"{WORK_DIR}/cache"
expected = [
    "m1_emb_musique_val_None.npy",
    "graph_v2_musique_val_None_m1.pkl",
    "edge_vecs_musique_val_None_m1.pkl",
]

print("Cache status:")
missing = []
for fname in expected:
    path = f"{cache_dir}/{fname}"
    if os.path.exists(path):
        print(f"  OK  {fname:50s}  {os.path.getsize(path)/1e6:7.1f} MB")
    else:
        print(f"  --  {fname:50s}  NOT FOUND")
        missing.append(fname)

if missing:
    print("\n⚠  Caches missing — run eval_kaggle.ipynb first (~15 min).")
    print("   The diagnostic will rebuild them automatically, but it will take longer.")
else:
    print("\n✓  All caches present — diagnostic runs fast (no recomputation)")

---
## Run Diagnostic
~5 min if caches exist · ~25 min if rebuilding from scratch

In [ ]:
# 6. Run diagnostic
import os
os.chdir(WORK_DIR)

diag_file = "/kaggle/working/diagnostic.txt"
os.system(f"python diagnose_pipeline.py 2>&1 | tee {diag_file}")
print("\nDiagnostic complete")

In [ ]:
# 7. Display results
with open("/kaggle/working/diagnostic.txt") as f:
    print(f.read())